In [4]:
import json
from datetime import datetime, timezone
from json import dumps

import numpy as np
import pandas as pd
from dotenv import dotenv_values
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer


/home/leolchalot/Documents/Ipssi/nosql_architecture/dev/projet/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 1. Chargement ciblé des variables d'environnement
env_local_vars = dotenv_values(".env.local")
local_uri = env_local_vars.get("LOCAL_URI")
model_embed = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Test de la connexion locale
try:
    if not local_uri:
        raise ValueError("LOCAL_URI non trouvé dans .env.local")
    client = MongoClient(local_uri, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    db = client["securite_routiere"]
    print("Connexion locale établie avec succès")
except Exception as e:
    print(f"Erreur de connexion locale : {e}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7462.01it/s]


Connexion locale établie avec succès


In [7]:

# 2. Chargement des fichiers CSV (ou récupération depuis les collections brutes)
df_caract = pd.read_csv("datasets/2024/caracteristiques-2024.csv", sep=";", low_memory=False)
df_lieux = pd.read_csv("datasets/2024/lieux-2024.csv", sep=";", low_memory=False)
df_vehicules = pd.read_csv("datasets/2024/vehicules-2024.csv", sep=";", low_memory=False)
df_usagers = pd.read_csv("datasets/2024/usagers-2024.csv", sep=";", low_memory=False)

# Remplacement des valeurs NaN par None (converti en null dans MongoDB)
for df in [df_caract, df_lieux, df_vehicules, df_usagers]:
    df.replace({np.nan: None}, inplace=True)

# 3. Nettoyage et préparation des données caractéristiques
def parse_date(row):
    try:
        if pd.isna(row.get("an")) or pd.isna(row.get("mois")) or pd.isna(row.get("jour")):
            return None

        # Conversion sécurisée de an, mois, jour
        an = int(float(row["an"]))
        mois = int(float(row["mois"]))
        jour = int(float(row["jour"]))

        # Gestion de l'heure et des minutes
        hrmn_raw = row.get("hrmn")
        hour, minute = 0, 0
        
        if hrmn_raw is not None and not pd.isna(hrmn_raw):
            hrmn_str = str(hrmn_raw).strip()
            hrmn_str = hrmn_str.removesuffix(".0")

            if ":" in hrmn_str:
                parts = hrmn_str.split(":")
                hour = int(parts[0])
                minute = int(parts[1])
            elif hrmn_str.isdigit():
                hrmn_str = hrmn_str.zfill(4)
                hour = int(hrmn_str[:2])
                minute = int(hrmn_str[2:])

        # timezone.utc directement accessible via l'import
        return datetime(an, mois, jour, hour, minute, tzinfo=timezone.utc)
    except Exception as e:
        return None

def parse_coord(val):
    if val is None or pd.isna(val):
        return None
    try:
        return float(str(val).replace(",", "."))
    except (ValueError, TypeError):
        return None

def generer_resume_accident(doc):
    """Traduit les données de l'accident en texte descriptif."""
    date_str = doc.get("hrmn", "")
    commune = doc.get("com", "inconnue")
    dep = doc.get("dep", "")
    
    nb_vehicules = len(doc.get("vehicules", []))
    nb_usagers = sum(len(v.get("usagers", [])) for v in doc.get("vehicules", []))
    
    # Construction du texte narratif
    texte = (
        f"Accident survenu dans le département {dep} (commune {commune}) à {date_str}. "
        f"Nombre de véhicules impliqués : {nb_vehicules}, total d'usagers : {nb_usagers}. "
        f"Type de collision : code {doc.get('col')}, météo : code {doc.get('atm')}, luminosité : code {doc.get('lum')}. "
        f"Voie : {doc.get('lieu', {}).get('voie', 'non précisée')}, surface de la route : {doc.get('lieu', {}).get('surf')}."
    )
    return texte

df_caract["date"] = df_caract.apply(parse_date, axis=1)
df_caract["lat_clean"] = df_caract["lat"].apply(parse_coord)
df_caract["long_clean"] = df_caract["long"].apply(parse_coord)

# 4. Structuration hiérarchique : Usagers -> Véhicules -> Accidents

# A. Dictionnaire des Usagers indexé par id_vehicule
usagers_by_vehicule = {}
cols_usager = [
    "id_usager", "place", "catu", "grav", "sexe", "An_nais",
    "trajet", "secu1", "secu2", "secu3", "locp", "actp", "etatp"
]
# Prise en compte du format secu1-3 si déjà agrégé
if "secu1-3" in df_usagers.columns:
    cols_usager = ["id_usager", "place", "catu", "grav", "sexe", "An_nais", "trajet", "secu1-3", "locp", "actp", "etatp"]

for row in df_usagers.to_dict(orient="records"):
    id_v = row.get("id_vehicule")
    doc_usager = {k: row[k] for k in cols_usager if k in row and row[k] is not None}
    usagers_by_vehicule.setdefault(id_v, []).append(doc_usager)

# B. Dictionnaire des Véhicules indexé par Num_Acc
vehicules_by_acc = {}
cols_vehicule = [
    "id_vehicule", "Num_Veh", "senc", "catv", "obs",
    "obsm", "choc", "manv", "motor", "occutc"
]

for row in df_vehicules.to_dict(orient="records"):
    num_acc = row.get("Num_Acc")
    id_v = row.get("id_vehicule")
    doc_vehicule = {k: row[k] for k in cols_vehicule if k in row and row[k] is not None}
    doc_vehicule["usagers"] = usagers_by_vehicule.get(id_v, [])
    vehicules_by_acc.setdefault(num_acc, []).append(doc_vehicule)

# C. Dictionnaire des Lieux indexé par Num_Acc
cols_lieu = [
    "catr", "voie", "V1", "V2", "circ", "nbv", "vosp", "prof",
    "pr", "pr1", "plan", "larrout", "surf", "infra", "situ", "vma"
]
lieux_by_acc = {}
for row in df_lieux.to_dict(orient="records"):
    num_acc = row.get("Num_Acc")
    doc_lieu = {k: row[k] for k in cols_lieu if k in row and row[k] is not None}
    lieux_by_acc[num_acc] = doc_lieu

# D. Construction finale des documents Accidents
cols_caract_env = ["adr", "dep", "com", "agg", "int", "atm", "lum", "col"]
accidents_docs = []

for row in df_caract.to_dict(orient="records"):
    if row.get("long_clean") is not None and row.get("lat_clean") is not None:
        geojson_location = {
            "type": "Point",
            "coordinates": [row.get("long_clean"), row.get("lat_clean")] # [longitude, latitude]
        }
    else:
        geojson_location = None

    num_acc = row.get("Num_Acc")
    doc_accident = {
        "Num_Acc": num_acc,
        "date": row.get("date"),
        "jour": row.get("jour"),
        "mois": row.get("mois"),
        "an": row.get("an"),
        "hrmn": str(row.get("hrmn")),
        "localisation": geojson_location,
        "lieu": lieux_by_acc.get(num_acc, {}),
        "vehicules": vehicules_by_acc.get(num_acc, [])
    }
    
    for k in cols_caract_env:
        if k in row and row[k] is not None:
            doc_accident[k] = row[k]
            
    accidents_docs.append(doc_accident)

# 5. Enrichissement vectoriel et écriture dans MongoDB
col_accidents = db["accidents"]
col_accidents.drop()  # Réinitialisation de la collection

if accidents_docs:
    print("1. Génération des résumés textuels...")
    textes = [generer_resume_accident(doc) for doc in accidents_docs]

    print("2. Calcul des embeddings (vectorisation par lot)...")
    # batch_size accélère le calcul si le jeu de données est volumineux
    vecteurs = model_embed.encode(textes, batch_size=64, show_progress_bar=True).tolist()

    print("3. Ajout des champs aux documents...")
    for i, doc in enumerate(accidents_docs):
        doc["resume_texte"] = textes[i]
        doc["embedding"] = vecteurs[i]

    print("4. Ingestion dans MongoDB...")
    col_accidents.insert_many(accidents_docs)
    
    # Création des index
    col_accidents.create_index("Num_Acc", unique=True)
    col_accidents.create_index("dep")
    col_accidents.create_index([("localisation", "2dsphere")])
    
    print(f"Succès : {len(accidents_docs)} accidents enrichis et insérés.")

1. Génération des résumés textuels...
2. Calcul des embeddings (vectorisation par lot)...


Batches: 100%|██████████| 851/851 [00:24<00:00, 35.46it/s]


3. Ajout des champs aux documents...
4. Ingestion dans MongoDB...
Succès : 54402 accidents enrichis et insérés.


In [5]:
client.close()